<div style="text-align: center;">
    <img src="./assets/ki_ag_main.jpg" alt="Logo" style="width: 1000px;"/>
</div>

<center><h1>Termin 4: Frontend Integration</h1></center>

<center>
    Heute geben wir unserer KI ein Gesicht! Wir haben in den letzten Wochen Trainingsdaten generiert, Modelle fine-getuned und einen Backend-Workflow in n8n gebaut.<br>
    Jetzt verbinden wir alles mit einer Web-Oberfläche und lernen dabei, wie Frontend und Backend kommunizieren.
</center>

# Kapitel 1: Die Schnittstelle öffnen (Webhook)

Bisher startet unser n8n Workflow nur, wenn wir manuell im internen Chat schreiben. Damit unsere eigene Webseite mit der KI sprechen kann, müssen wir den Workflow **von außen erreichbar** machen. Dafür nutzen wir einen **Webhook**.

### Was ist ein Webhook?

Ein Webhook ist wie eine "Türklingel" für deinen Workflow:
- Jemand (z.B. eine Webseite) klingelt an einer bestimmten URL
- n8n "hört" das und startet den Workflow
- Am Ende schickt n8n eine Antwort zurück

### Aufgabe 1.1: Chat-Trigger durch Webhook ersetzen

1. Öffne deinen n8n Workflow im Browser
2. **Lösche** die Node "When chat message received" (Chat Trigger)
3. Klicke oben rechts auf das **+** Symbol
4. Suche nach **"Webhook"** und füge die Node hinzu
5. Verbinde sie mit dem Start deiner Logik (z.B. dem Memory Manager)

### Aufgabe 1.2: Webhook konfigurieren

Doppelklicke auf die Webhook-Node und konfiguriere sie:

| Einstellung | Wert | Erklärung |
|-------------|------|-----------|
| HTTP Method | `POST` | Wir **senden** Daten an den Server |
| Path | `host` | Der Name der Ressource |
| Respond | `When Last Node Finishes` | Antwort erst wenn alles fertig ist |

> ⚠️ **WICHTIG:** Klicke oben in der Node auf den Reiter **Test URL** (nicht Production URL) und lass das Fenster offen.

# Kapitel 2: Den Webhook testen (PowerShell)

Bevor wir eine Webseite bauen, müssen wir sicherstellen, dass die "Leitung steht". Wir simulieren einen Webseiten-Aufruf über die **Windows PowerShell**.


### Aufgabe 2.1: n8n Webhook aktivieren

1. Gehe zurück zu deiner Webhook-Node in n8n
2. Klicke auf den großen roten Button: **`Listen for test event`**
3. n8n wartet jetzt auf ein Signal von außen...

### Aufgabe 2.2: Test-Request senden

1. Öffne die **Windows PowerShell** (Start → "PowerShell" eingeben)
2. Kopiere den folgenden Code
3. Drücke Enter

In [ ]:
# In die Windows PowerShell kopieren:
$body = @{ chatInput = "Hallo ich bin hier für das Assessment Center" } | ConvertTo-Json
Invoke-RestMethod -Uri "http://localhost:5678/webhook-test/host" -Method Post -Body $body -ContentType "application/json"

### Aufgabe 2.3: Webhook-Variable anpassen

Vermutlich wirst du nach dem ersten Test eine **Fehlermeldung** erhalten. Das ist zu erwarten!

**Das Problem:** Unsere Nodes im Workflow warten noch auf die Node "When Chat Message Received", die wir entfernt haben. Sie versuchen den User-Input von dort zu holen – aber diese Node existiert nicht mehr.

**Die Lösung:** Ersetze in allen betroffenen Nodes die alte Variable durch die neue Webhook-Variable.

| Alt (funktioniert nicht mehr) | Neu (vom Webhook) |
|------------------------------|-------------------|
| `{{ $('When chat message received').first().json.chatInput }}` | `{{ $('Webhook').first().json.body.chatInput }}` |

**So gehst du vor:**

1. Öffne jede Node, die den Chat-Input verwendet 
2. Suche nach `$('When chat message received').first().json.chatInput`
3. Ersetze es durch `$('Webhook').first().json.body.chatInput`
4. Speichere die Node

### Erwartetes Ergebnis

**In der PowerShell:** Du siehst die Antwort der KI (falls nicht liegt es daran, dass n8n den Datenfluss unterbricht, dann musst du in n8n den Datenfluss manuell weiterführen mit `Execute Step` Buttons innerhalb der Nodes)

**In n8n:** Die Webhook-Node wird grün ("Success") und du kannst den Datenfluss verfolgen

### Häufige Fehler

| Fehler | Ursache | Lösung |
|--------|---------|--------|
| `404 Not registered` | Workflow nicht aktiv | Klicke auf "Listen for test event" |
| `Method Not Allowed` | Falsche HTTP-Methode | Stelle Webhook auf `POST` |
| `Cannot read property` | Alte Variable | Aufgabe 2.3: Variable anpassen |
| `Workflow wird nur erste Node ausgelöst` | n8n spezifisches Problem | manuelles weiterklicken notwendig in den einzelnen Nodes zum testen |

# Kapitel 3: Das Vue.js Projekt starten

Wir nutzen **Vue.js**, ein modernes JavaScript Framework, für unsere Benutzeroberfläche

### Aufgabe 3.1: Projekt entpacken

1. Entpacke die Datei `assessment-center.zip` welche im heruntergeladenen Ordner `04_UI` liegt.
2. Du solltest jetzt einen Ordner `assessment-center` haben

### Aufgabe 3.2: Dependencies installieren

1. Öffne ein Terminal / PowerShell
2. Navigiere in den Ordner:

In [ ]:
# Im Terminal ausführen:

cd assessment-center
npm install

Das dauert ca. 1-2 Minuten. npm lädt alle benötigten Pakete herunter.

### Aufgabe 3.3: Entwicklungsserver starten

In [ ]:
# Nach der Installation:

npm run dev

Du solltest sehen:
```
  VITE v7.x.x  ready in xxx ms

  ➜  Local:   http://localhost:5173/
```

Öffne jetzt **http://localhost:5173** im Browser. Du solltest das Chat-Interface sehen!

# Kapitel 4: Projektstruktur verstehen

Bevor wir weitermachen, schauen wir uns an, wie das Projekt aufgebaut ist.

### Ordnerstruktur

```
assessment-center/
├── src/
│   ├── components/           # Wiederverwendbare UI-Bausteine
│   │   ├── ChatWindow.vue    # Container für alle Nachrichten
│   │   ├── MessageBubble.vue # Einzelne Chat-Nachricht
│   │   ├── ChatInput.vue     # Eingabefeld + Button
│   │   └── ResultCard.vue    # Ergebnis-Anzeige
│   ├── composables/
│   │   └── useAssessment.js  # Logik & State
│   ├── services/
│   │   └── api.js            # Schnittstellen zum n8n-Backend
│   ├── App.vue               # Root Component orchestriert Anwendung
│   ├── main.js               # App-Start
│   └── style.css             # Globale Styles
├── index.html                # HTML-Grundgerüst
└── package.json              # Projekt-Konfiguration
```

### Die wichtigsten Dateien

| Datei | Aufgabe |
|-------|--------|
| `src/services/api.js` | **Konfiguration** Hier müssen die n8n Webhook-URLs eingetragen werden. |
| `src/composables/useAssessment.js` | **Logik und State** Beinhaltet die Steuerung der Phasen (host, angry, scorer) und verwaltet den Chatverlauf und State (chatHistory, currentPhase) |
| `src/App.vue` | **Integration** Verbindet die Logik aus dem Composable mit den UI-Komponenten. |
| `src/components/*.vue` | Die UI-Bausteine |

### Aufgabe 4.1: Code erkunden

Öffne das Projekt in VS Code und beantworte diese Fragen:

1. In welcher Datei ist der State `chatHistory` definiert?
2. Welche Props erwartet die MessageBubble Komponente?
3. Was passiert in der `sendMessage()` Funktion in api.js?
4. In der Funktion `useAssessment()` in `useAssessment.js` werden reaktive Objekte mittels `const currentPhase = ref('host')` erstellt. Was sind reaktive Objekte in Vue und warum verwendet man diese?

In [ ]:
# Deine Antworten:

# 1. chatHistory ist definiert in:


# 2. MessageBubble erwartet:


# 3. sendMessage() in api.js:


# 4. Was sind reaktive Objekte in Vue und warum verwendet man diese?:



# Kapitel 5: Webhook URLs eintragen

Jetzt verbinden wir Frontend und Backend!

### Aufgabe 5.1: api.js öffnen und URL eintragen

Öffne die Datei `src/services/api.js`. Du siehst dort:

In [ ]:
// In src/services/api.js:

const URLS = {
    host:   'http://localhost:5678/webhook-test/DEINE-HOST-ID', //hier eintragen
    angry:  'http://localhost:5678/webhook-test/DEINE-ANGRY-ID',
    scorer: 'http://localhost:5678/webhook-test/DEINE-SCORER-ID'
}

Für den Host tragen wir jetzt die URL ein, die wir davor definiert haben.

1. Gehe in n8n zu deiner Webhook-Node
2. Kopiere die Test URL
3. Ersetze `DEINE-HOST-ID` mit deiner URL
4. Speichere die Datei

### Aufgabe 5.2: Testen!

1. Stelle sicher, dass n8n auf "Listen for test event" wartet
2. Öffne http://localhost:5173 im Browser
3. Schreibe eine Nachricht und klicke "Senden"
4. Beobachte: Kommt eine Antwort?
5. Bei Fehlern F12 und dann in der Console prüfen (CORS Fehler nicht relevant)
6. Du kannst nun den Webhook von `Test URL` auf `Production URL switchen` um nicht immer erneut den Test aktivieren zu müssen, achte darauf, dass du oben rechts den Workflow auch von inactive auf active setzt und in `api.js` die URL entsprechend anpasst von `http://localhost:5678/webhook-test/host` auf `'http://localhost:5678/webhook/host'` (Wenn du den Workflow über den Production Webhook aufrufst, wirst du kein visuelles Feedback im Workflow mehr bekommen (keine Grünen Linien oder Input/Output) Mit der Production URL passieren auch weniger n8n spezifische Fehler wie das triggern von nur einer single Node)

# Kapitel 6: Workflows auftrennen (3 Phasen)

Aktuell haben wir einen riesigen "Monolith"-Workflow, der alles macht: Smalltalk, wütender Kunde, Scoring.

Das Problem:
- Schwer zu debuggen
- n8n "vergisst" manchmal den Kontext zwischen Phasen
- Keine klare Trennung der Verantwortlichkeiten

- **Frontend** speichert den Zustand (`currentPhase`) und den Chatverlauf (`chatHistory`)
- **3 kleine n8n Workflows** sind jeweils nur für eine Aufgabe zuständig

```
┌─────────────────────────────────────────────────────────────┐
│                        FRONTEND                             │
│   chatHistory: [...]     currentPhase: 'host'               │
└─────────────────────────────────────────────────────────────┘
         │                      │                      │
         ▼                      ▼                      ▼
   ┌──────────┐          ┌──────────┐          ┌──────────┐
   │  HOST    │          │  ANGRY   │          │  SCORER  │
   │ Workflow │          │ Workflow │          │ Workflow │
   └──────────┘          └──────────┘          └──────────┘
```

### Aufgabe 6.1: Workflows duplizieren

1. Öffne deinen bestehenden Workflow in n8n
2. Klicke oben rechts auf **...** → **Duplicate**
3. Wiederhole das, sodass du **3 Workflows** hast
4. Benenne sie:
   - `1_AC_Host`
   - `2_Angry_Customer`
   - `3_Scorer`

### Aufgabe 6.2: Workflow 1 - Der Host

**Webhook Pfad:** `/host`

**Aufgabe:** Nur der freundliche Smalltalk-Teil.

**Anpassungen:**
1. Lösche alle Nodes die mit dem Angry Customer und dem Scorer zusammenhängen
2. Behalte nur die Nodes die mit dem AC Host zusammenhängen
3. Die `If-Node` sollte nun mit dem `Chat Memory Manager 2` Node verbunden werden, diese löscht den Chatverlauf.
4. Füge danach eine `Edit Fields 1` Node hinzu mit zwei variablen:
* `output`: "Vielen Dank für die Vorstellung. Kommen wir nun zum praktischen Teil des Assessments. Wir simulieren jetzt eine Stresssituation im Kundenservice. Du bist der Support-Mitarbeiter und gleich wird sich ein unzufriedener Kunde melden. Versuche, das Problem professionell zu lösen. Viel Erfolg!"
* `nextPhase`: angry_customer

5. der Workflow sollte dann ungefähr so aussehen:

<div style="text-align: center;">
    <img src="./assets/ac_host_workflow.png" alt="Logo" style="width: 1000px;"/>
</div>

---

### Aufgabe 6.3: Workflow 2 - Angry Customer

**Webhook Pfad:** `/angry_customer`

**Aufgabe:** Nur der wütende Kunde-Teil.

**Anpassungen:**
1. Passe auch hier dementsprechend den Workflow an, wie zuvor beim host.
2. ändere in der hinzugefügten `Edit Fields 1` Node:
* `output`:  Die Simulation ist beendet. Vielen Dank! Deine Antworten werden jetzt analysiert. In kürze erhälst du deine Ergebnisse...
* `nextPhase` : scorer

---

### Aufgabe 6.4: Workflow 3 - Der Scorer

**Webhook Pfad:** `/scorer`

**Aufgabe:** Bekommt den gesamten Chatverlauf und bewertet.

**Wichtiger Unterschied:** Dieser Workflow erwartet keine einzelne Nachricht, sondern ein **Array mit dem kompletten Verlauf**!

Diesen bekommen wir vom Frontend gesendet.
Lösche nun die `Chat Memmory Manager` Node

**Input vom Frontend:**
```json
{
    "fullHistory": [
        { "sender": "bot", "text": "Willkommen..." },
        { "sender": "user", "text": "Hallo, ich bin Max" },
        ...
    ]
}
```

**In n8n:** Passe die Node `user input extrahieren an` und ersetze ihn mit folgendem:

In [ ]:
// Code durch diesen ersetzen:
const history = $input.first().json.body.fullHistory || [];

const userMessages = history.filter(msg => msg.sender === 'user');

return userMessages.map((msg, index) => {
  return {
    json: {
      messageIndex: index + 1,
      text: msg.text 
    }
  }
});

* Lösche die `Merge Node` diese wird nicht mehr benötigt, da es nur noch einen Datenstrom gibt und nicht mehr gewartet werden muss 
* Passe die letzte `Edit Fields` Node nun so an, dass sie als `output` speziell hierrauf zugreift: `{{ $('Zusammenfassung ResultHTTP Request').item.json.message.content }}`

Der scorer Workflow sollte dann in etwa so aussehen:

<div style="text-align: center;">
    <img src="./assets/scorer_workflows.png" alt="Logo" style="width: 1000px;"/>
</div>

### Aufgabe 6.5: Alle URLs eintragen

Nachdem du die 3 Workflows erstellt hast:

1. Kopiere die Production URL von jedem Workflow (Workflow auf active setzen für Production URL)
2. Trage sie in `src/services/api.js` ein:

```javascript
const URLS = {
    host:   'http://localhost:5678/webhook/abc123...',  // Deine Host URL
    angry:  'http://localhost:5678/webhook/def456...',  // Deine Angry URL
    scorer: 'http://localhost:5678/webhook/ghi789...'   // Deine Scorer URL
}
```

### Aufgabe 6.6: Komplett-Test

1. Setze alle 3 n8n Workflows auf active und verwende den Produktion Webhook Pfad (bereits in 6.5 erledigt). 
2. Öffne http://localhost:5173 im Browser
3. Führe ein komplettes Gespräch:
   - 6 Nachrichten mit dem Host (Badge zeigt "Interview")
   - 4 Nachrichten mit dem Angry Customer (Badge zeigt "Kundenservice")
   - Ergebnis anschauen! (Badge zeigt "Abgeschlossen")

# Kapitel 7: Bonus - Eigene Anpassungen

Wenn du noch Zeit hast, hier ein paar Ideen:

### 7.1 Switchen der Phasen dynamisch anhand von LLM Analyse

Im n8n Workflow des hosts 1_AC_Host ändere die Logik so, dass nicht nach einer bestimmten Anzahl an messages, sondern nachdem der Host alle notwendigen Informationen erhalten hat, erst zum angry Customer geswitched wird.

**Hinweise:**

1. Dafür ist ein weiterer LLM Aufruf notwendig, der einen Systemprompt übergeben bekommen sollte mit der Anweisung explizit zu analysieren ob schon alles genannt wurde. Das LLM muss auch die Chat History übergeben bekommen. Das LLM sollte als Output nur eine binäre Antwort ausgeben True/False und evtl. in Json Format
2. Die `If-Node` muss dafür auch angepasst werden und nach True/False vergleichen statt zahlenbasiert.
3. Flüssigerer Übergang zur nächsten Phase, indem man die ac_host persona überleiten lässt, anstatt den default Text in `Edit Fields`

### 7.2: Styling anpassen

Du kannst die Componenten, Styling und Layout beliebig anpassen und die Anwendung individualisieren. Mache es zu deinem Design

### 7.3: Stellenanzeige hinzufügen

Erweitere das Frontend so, dass der User am Anfang eine Stellenanzeige eingeben kann:
1. Füge ein `textarea` hinzu
2. Speichere den Text in einer Variable
3. Sende ihn an einen neuen Workflow 04_Stellenanzeige in n8n. 
4. n8n bietet die Möglichkeit mit einer `HTTP Request` Node auf Website links zuzugreifen und mit einer `HTML Extract` Node ausgewählte Abschnitte einer Website zu extrahieren. 
5. Diese können dann einem LLM Übergeben werden, welches einen Systemprompt für den ac_host später erstellt 
6. Der Systemprompt muss vom Frontend gespeichert werden und dann dem ac_host im API-Aufruf mitgegeben werden. Dafür muss dann auch der 1_AC_Host Workflow angepasst werden
7. (Hinweis: Extraktion per Link eignet sich nicht bei jeder Stellenanzeige, jedoch bei einigen wie z.B. Mercedes)

# Zusammenfassung

### Architektur-Überblick

```
┌────────────────────────────────────────────────────────────────┐
│                    BROWSER (Vue.js)                            │
│  ┌──────────────────────────────────────────────────────────┐  │
│  │  State:                                                  │  │
│  │  - chatHistory: Array mit allen Nachrichten              │  │
│  │  - currentPhase: 'host' | 'angry' | 'scorer' | 'result'  │  │              
│  └──────────────────────────────────────────────────────────┘  │
└────────────────────────────────────────────────────────────────┘
                              │
                    fetch() POST Requests
                              │
                              ▼
┌────────────────────────────────────────────────────────────────┐
│                          n8n                                   │
│  ┌────────────┐  ┌────────────┐  ┌────────────┐              │
│  │ 1_Host     │  │ 2_Angry    │  │ 3_Scorer   │              │
│  │ /host      │  │ /angry     │  │ /scorer    │              │
│  └────────────┘  └────────────┘  └────────────┘              │
└────────────────────────────────────────────────────────────────┘
                              │
                      HTTP Request
                              │
                              ▼
┌────────────────────────────────────────────────────────────────┐
│                      Ollama (LLM)                              │
│                  Fine-tuned Modelle                            │
└────────────────────────────────────────────────────────────────┘
```

---

### Glückwunsch!

Du hast in 4 Terminen eine komplette KI-Anwendung gebaut:

1. **Termin 1:** Lokale LLMs verstehen und synthetische Datenpipeline(Ollama)
2. **Termin 2:** Fine-Tuning mit eigenen Daten (LoRA)
3. **Termin 3:** Backend-Orchestrierung (n8n)
4. **Termin 4:** Frontend-Integration (Vue.js)